In [1]:
import torch 

/home/mnl/Desktop/University/Fall 2025-2026/fyp/K-KANs/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


### True Data Generation Process 

In [112]:
torch.manual_seed(0)
n = 150
p = 10 #NOTE: so that the true covariance matrix is not full rank (since n > p) (if we use a linear kernel without a jitter)

## we do not really care where X comes from
X_data = torch.distributions.multivariate_normal.MultivariateNormal(
    loc= torch.zeros(p) , # mean vector of zeros
    covariance_matrix=torch.eye(p)  # identity covariance matrix
).sample((n,))  # n x p

## what we care about is that the vector (y_1, ..., y_n)^T comes from a guassian process with a certain kernel (that depends on X)
## in this example I will be taking that k(x_i,x_j) = linear kernel = x_i^T * x_j, where the sigma^2 = 1
true_mean = torch.zeros(n)  # n x 1
true_cov = torch.eye(n)  # n x n
true_sigma_squared = 1.0 
l_for_rbf = 1.0  # length scale for rbf kernel
for i in range(n):
    for j in range(n):
        # rbf kernel
        true_cov[i, j] = true_sigma_squared * torch.exp(-torch.norm(X_data[i] - X_data[j])**2 / (2 * (l_for_rbf ** 2))) #NOTE: even if I used this kernel, the determinant is being 0, but the rank is full.
        # true_cov[i, j] = true_sigma_squared * (X_data[i].unsqueeze(0) @ X_data[j].unsqueeze(1))   # linear kernel, we willdo  unsqueeze to add a dimension for matrix multiplication, resulting in a (1 x p) @ (p x 1) = (1 x 1) tensor
        # # adding a small value to the diagonal to ensure positive definiteness #NOTE(see README for more details)
        # # true_cov[i, i] += torch.distributions.normal.Normal(0, 0.001).sample()
        # true_cov[i, i] += 1e-5 #NOTE this jitter is very weird, I am not sure why it is working, #NOTE: not always it works

rank = torch.linalg.matrix_rank(true_cov)
print(f"rank(true_cov): {rank}, expected: {n}")  # should be n
print(f"Determinant(true_cov): {torch.linalg.det(true_cov)}, expected: non-zero" )  # should be non-zero, #NOTE: if we get 0 this MIGHT mean underflow 

## by the assumption of the guassian process, the vector (f(x_1), ..., f(x_n))^T follows a multivariate normal distribution with mean 0 and covariance matrix K (n x n)
## so we can sample from that distribution to get f_data = (f(x_1), ..., f(x_n))^T
f_data = torch.distributions.multivariate_normal.MultivariateNormal(
    loc=true_mean.squeeze(),  # n (not n x 1)
    covariance_matrix=true_cov
).sample()  # n x 1
# make f_data of shape (n, 1)
f_data = f_data.unsqueeze(1)  # n x 1
print(f"Shape of f_data: {f_data.shape}, expected: ({n}, 1)")  # should be (n, 1)

## for this first experiment, we will assume y = f(x) without noise
y_data = f_data  # n x 1
print(f"Shape of y_data: {y_data.shape}, expected: ({n}, 1)")  # should be (n, 1)

rank(true_cov): 150, expected: 150
Determinant(true_cov): 0.04505434259772301, expected: non-zero
Shape of f_data: torch.Size([150, 1]), expected: (150, 1)
Shape of y_data: torch.Size([150, 1]), expected: (150, 1)


In [113]:
# now my goal is to maximize the marginal log likelihood, so I will define my loss = - MLL (because we care to maximize MLL which is equivalent to minimizing - MLL)

## intialization of the kernel matrix
sigmas_pred = torch.randn(p,1, requires_grad=True)
learning_rate = 0.001
epochs = 500
losses = []
for epoch in range(epochs): 
    linear_kernel_pred = torch.ones(n,n) # these we do not care about their value they are just placeholder for now #TODO: investigate if the gradients will track chnages from one to k_i_j or not ( I think no )
    # instead of this loop;;;; I will do matrix multiplication 
    for i in range(n):
        for j in range(n): 
            k_i_j = 0
            for k in range(p):
                k_i_j += sigmas_pred[k]**2 * X_data[i,k] * X_data[j,k]
            linear_kernel_pred[i,j] = k_i_j + 1e-3  # adding jitter to ensure positive definiteness
    # print(f"Rank of predicted kernel at epoch {epoch} is {torch.linalg.matrix_rank(linear_kernel_pred)}, expected: {n}")  # should be n
    # print(f"Determinant of predicted kernel at epoch {epoch} is {torch.linalg.det(linear_kernel_pred)}, expected: non-zero")  # should be non-zero
    # Computation of the loss 
    K = linear_kernel_pred 
    K_inv = torch.linalg.inv(K)
    mu = torch.zeros(n,1) #TODO: how can we make it depend also on X_data
    sub = y_data - mu
    #NOTE: if we do it this way it will become infinity becasue of the exponentiation
    # num = torch.exp(-0.5* (sub.T @ K_inv @ sub)) 
    # den = torch.sqrt((2*torch.pi)**n * torch.det(K))  
    # MLL = num/den
    # loss = - torch.log(MLL + 1e-10)  # adding a small value to avoid log(0) 
    quad = (sub.T @ K_inv @ sub).squeeze()            # (y-mu)^T K^{-1} (y-mu)
    logdet = torch.logdet(K)                          # log det(K)
    log_mll = -0.5 * quad - 0.5 * logdet - 0.5 * n * torch.log(torch.Tensor([2])*torch.pi)
    loss = -log_mll
    losses.append(loss.item())

    loss.backward()
    print(f" Loss at epoch {epoch} is {loss.item()}")
    # update sigmas_pred
    with torch.no_grad():
        sigmas_pred -= learning_rate * sigmas_pred.grad # p x 1 
        sigmas_pred.grad.zero_()

    if epoch % 10 == 0: 
        print(f"At epoch {epoch}, Covariance Matrix Predicted is : {K}")
        print(f"Loss at epoch {epoch} is {loss.item()}")
print(f" Losses over epochs: {losses}")


 Loss at epoch 0 is nan
At epoch 0, Covariance Matrix Predicted is : tensor([[ 18.6017,  11.2591,   2.1673,  ..., -15.7369,   5.3775,  -8.3363],
        [ 11.2591,  23.1090,  10.8673,  ...,   3.5512,   4.9474,  -5.4175],
        [  2.1673,  10.8673,  23.8431,  ...,  -2.7875,   4.7363,  -8.6271],
        ...,
        [-15.7369,   3.5512,  -2.7875,  ..., 127.4454,   1.7101,  36.9970],
        [  5.3775,   4.9474,   4.7363,  ...,   1.7101,  18.8514,  -4.4068],
        [ -8.3363,  -5.4175,  -8.6271,  ...,  36.9970,  -4.4068,  22.9297]],
       grad_fn=<CopySlices>)
Loss at epoch 0 is nan
 Loss at epoch 1 is 1098.971435546875
 Loss at epoch 2 is 1098.971435546875
 Loss at epoch 3 is 1098.971435546875
 Loss at epoch 4 is 1098.971435546875
 Loss at epoch 5 is 1098.971435546875
 Loss at epoch 6 is 1098.971435546875
 Loss at epoch 7 is 1098.971435546875
 Loss at epoch 8 is 1098.971435546875


KeyboardInterrupt: 